#Gold Layer - For Reporting and advanced analytics
1. Windowing
2. Aggregation

In [1]:
#Run the below notebook for invoking the function to this notebook
%run "/notebook/PysparkCode/1_Generic_function.ipynb"

In [2]:
df_Enhanced=spark.read.orc("/dataset/Silver")
#df_Enhanced.show()


+------+--------------------+------------+--------------+--------------------+--------------------+-------------+-----+--------------------+-----------------------------+-----------------+---------------+----------------+--------------------+--------------------+--------------------+-------------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+-------------------+------------+--------------+-------------+
|Job ID|              Agency|Posting Type|# Of Positions|      Business Title| Civil Service Title|Title Code No|Level|        Job Category|Full-Time/Part-Time indicator|Salary Range From|Salary Range To|Salary Frequency|       Work Location|  Division/Work Unit|     Job Description|Minimum Qual Requirements|    Preferred Skills|Additional Information|            To Apply|         Hours/Shift|     Work Location

In [3]:
#Whats the job posting having the highest salary per agency?
from pyspark.sql.window import Window
from pyspark.sql.functions import *

def high_sal(df):
    df_highestSal=df.withColumn("Rank",rank().over(Window.partitionBy("Posting Type","Agency").orderBy(desc("Annual_salary"))))\
    .select("Posting Type","Agency","Annual_salary","Rank")
    return df_highestSal

high_sal(df_Enhanced).where("Rank = 1").show()

+------------+--------------------+------------------+----+
|Posting Type|              Agency|     Annual_salary|Rank|
+------------+--------------------+------------------+----+
|    Internal|DEPT OF CITYWIDE ...|          140171.2|   1|
|    Internal|TEACHERS RETIREME...|           69078.5|   1|
|    Internal|BOROUGH PRESIDENT...|           80526.5|   1|
|    External|HRA/DEPT OF SOCIA...|          100000.0|   1|
|    Internal|DEPARTMENT OF PRO...|          127500.0|   1|
|    Internal|OFF OF PAYROLL AD...|           85995.0|   1|
|    External|DEPARTMENT OF INV...|          165000.0|   1|
|    Internal|DEPARTMENT OF BUS...|          111377.0|   1|
|    External|DEPT OF CITYWIDE ...|          140171.2|   1|
|    External|DEPARTMENT OF PRO...|          127500.0|   1|
|    Internal|DEPARTMENT OF FIN...|          125000.0|   1|
|    External|DEPARTMENT OF FIN...|          106019.0|   1|
|    External|BOROUGH PRESIDENT...|           80526.5|   1|
|    Internal|DEPARTMENT OF COR...|13910

In [4]:
#Whats the job positings average salary per agency for the last 2 years?
from pyspark.sql.window import Window
from pyspark.sql.functions import *

def avg_sal(df):
    avg_sal=df.where("year(`Posting Date`) >= year(current_date())-2 ").groupBy("Posting Type","Agency").agg(avg(col("Annual_salary")).alias("avg_sal"))
    return avg_sal

avg_sal(df_Enhanced).show()


+------------+------+-------+
|Posting Type|Agency|avg_sal|
+------------+------+-------+
+------------+------+-------+



In [5]:
#What are the highest paid skills in the US market?
from pyspark.sql.window import Window

def high_paid(df):
    df1=df.groupBy("Division/Work Unit").agg(max(col("Annual_salary")).alias("highest_paid")).orderBy("highest_paid",ascending=False)
    return df1

high_paid(df_Enhanced).show(1)


+--------------------+------------+
|  Division/Work Unit|highest_paid|
+--------------------+------------+
|BCS - Executive O...|    218587.0|
+--------------------+------------+
only showing top 1 row

